In [3]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

In [7]:
from langchain_core.documents import Document
doc1 = Document(
        page_content="Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.",
        metadata={"team": "Royal Challengers Bangalore"}
    )
doc2 = Document(
        page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.",
        metadata={"team": "Mumbai Indians"}
    )
doc3 = Document( 
        page_content="MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.",
        metadata={"team": "Chennai Super Kings"}
    )
doc4 = Document(
        page_content="Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.",
        metadata={"team": "Mumbai Indians"}
    )
doc5 = Document(
        page_content="Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.",
        metadata={"team": "Chennai Super Kings"}
    )

In [5]:
docs = [doc1, doc2, doc3, doc4, doc5]

In [10]:
vector_store = Chroma(
    embedding_function=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2"),
    persist_directory='my_chroma_db',
    collection_name='sample'
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4999.81it/s]


In [11]:
vector_store.add_documents(docs)

['74a74592-94ac-4c0d-a36d-e1f0061b352d',
 '729c6795-4abf-434e-9232-4f709fbfaf85',
 '1310b5a1-5a66-43f9-91cb-3fb9f834279b',
 '7161fb4e-649a-46f7-85b8-f1a6ab3d7d1f',
 'a4796671-3478-4243-b859-7558e100c8eb']

In [12]:
vector_store.get(include=['embeddings','documents', 'metadatas'])

{'ids': ['74a74592-94ac-4c0d-a36d-e1f0061b352d',
  '729c6795-4abf-434e-9232-4f709fbfaf85',
  '1310b5a1-5a66-43f9-91cb-3fb9f834279b',
  '7161fb4e-649a-46f7-85b8-f1a6ab3d7d1f',
  'a4796671-3478-4243-b859-7558e100c8eb'],
 'embeddings': array([[ 0.00994727,  0.06914337, -0.05147117, ..., -0.03543334,
          0.01284807,  0.0124829 ],
        [ 0.00127744,  0.03129853, -0.02375376, ..., -0.00518358,
         -0.03280612,  0.02737714],
        [-0.10265914,  0.02650812,  0.02271502, ..., -0.03359744,
         -0.07984945, -0.01507709],
        [ 0.02123396, -0.02468546, -0.04494375, ..., -0.10995811,
          0.00572558,  0.09915376],
        [ 0.01873978,  0.04382848, -0.04304255, ..., -0.07801619,
         -0.0784068 , -0.00304187]], shape=(5, 384)),
 'documents': ['Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.',
  "Rohit Sharma is the mo

In [13]:
vector_store.similarity_search(
    query='Who among these are a bowler?',
    k=2
)

[Document(id='7161fb4e-649a-46f7-85b8-f1a6ab3d7d1f', metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
 Document(id='729c6795-4abf-434e-9232-4f709fbfaf85', metadata={'team': 'Mumbai Indians'}, page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.")]

In [14]:
# search with similarity score
'''
diff between similarity_search and similarity_search_with_scoreThe core difference is that similarity_search only returns a list of matching documents, 
while similarity_search_with_score returns both the documents and their numerical similarity scores.
'''
vector_store.similarity_search_with_score(
    query='Who among these are a bowler?',
    k=2
)

[(Document(id='7161fb4e-649a-46f7-85b8-f1a6ab3d7d1f', metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
  0.9693599939346313),
 (Document(id='729c6795-4abf-434e-9232-4f709fbfaf85', metadata={'team': 'Mumbai Indians'}, page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure."),
  1.14934504032135)]

In [15]:
# meta-data filtering
vector_store.similarity_search_with_score(
    query="",
    filter={"team": "Chennai Super Kings"}
)

[(Document(id='1310b5a1-5a66-43f9-91cb-3fb9f834279b', metadata={'team': 'Chennai Super Kings'}, page_content='MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.'),
  1.8436006307601929),
 (Document(id='a4796671-3478-4243-b859-7558e100c8eb', metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.'),
  1.8909372091293335)]

In [16]:
# update documents
updated_doc1 = Document(
    page_content="Virat Kohli, the former captain of Royal Challengers Bangalore (RCB), is renowned for his aggressive leadership and consistent batting performances. He holds the record for the most runs in IPL history, including multiple centuries in a single season. Despite RCB not winning an IPL title under his captaincy, Kohli's passion and fitness set a benchmark for the league. His ability to chase targets and anchor innings has made him one of the most dependable players in T20 cricket.",
    metadata={"team": "Royal Challengers Bangalore"}
)
vector_store.update_document(document_id='09a39dc6-3ba6-4ea7-927e-fdda591da5e4', document=updated_doc1)


In [17]:
# view documents
vector_store.get(include=['embeddings','documents', 'metadatas'])

{'ids': ['74a74592-94ac-4c0d-a36d-e1f0061b352d',
  '729c6795-4abf-434e-9232-4f709fbfaf85',
  '1310b5a1-5a66-43f9-91cb-3fb9f834279b',
  '7161fb4e-649a-46f7-85b8-f1a6ab3d7d1f',
  'a4796671-3478-4243-b859-7558e100c8eb'],
 'embeddings': array([[ 0.00994727,  0.06914337, -0.05147117, ..., -0.03543334,
          0.01284807,  0.0124829 ],
        [ 0.00127744,  0.03129853, -0.02375376, ..., -0.00518358,
         -0.03280612,  0.02737714],
        [-0.10265914,  0.02650812,  0.02271502, ..., -0.03359744,
         -0.07984945, -0.01507709],
        [ 0.02123396, -0.02468546, -0.04494375, ..., -0.10995811,
          0.00572558,  0.09915376],
        [ 0.01873978,  0.04382848, -0.04304255, ..., -0.07801619,
         -0.0784068 , -0.00304187]], shape=(5, 384)),
 'documents': ['Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.',
  "Rohit Sharma is the mo

In [18]:
# delete document
vector_store.delete(ids=['09a39dc6-3ba6-4ea7-927e-fdda591da5e4'])

In [19]:
# view documents
vector_store.get(include=['embeddings','documents', 'metadatas'])

{'ids': ['74a74592-94ac-4c0d-a36d-e1f0061b352d',
  '729c6795-4abf-434e-9232-4f709fbfaf85',
  '1310b5a1-5a66-43f9-91cb-3fb9f834279b',
  '7161fb4e-649a-46f7-85b8-f1a6ab3d7d1f',
  'a4796671-3478-4243-b859-7558e100c8eb'],
 'embeddings': array([[ 0.00994727,  0.06914337, -0.05147117, ..., -0.03543334,
          0.01284807,  0.0124829 ],
        [ 0.00127744,  0.03129853, -0.02375376, ..., -0.00518358,
         -0.03280612,  0.02737714],
        [-0.10265914,  0.02650812,  0.02271502, ..., -0.03359744,
         -0.07984945, -0.01507709],
        [ 0.02123396, -0.02468546, -0.04494375, ..., -0.10995811,
          0.00572558,  0.09915376],
        [ 0.01873978,  0.04382848, -0.04304255, ..., -0.07801619,
         -0.0784068 , -0.00304187]], shape=(5, 384)),
 'documents': ['Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.',
  "Rohit Sharma is the mo